# Assignment 3 — PyTorch Neural Network Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/PML/blob/main/code-scripts/T8_torch_NN_pipeline_and_questions_for_assign_3_answered.ipynb)

This notebook is based on the PyTorch neural network pipeline notebook for Assignment 3.

The goal is to edit the notebook and answer the four questions about:

1. PyTorch `DataLoader` objects and tensor shapes.
2. Model architecture changes for different input sizes and hidden layers.
3. Training time comparison with and without `num_workers=2`.
4. Train and validation accuracy curves over epochs.

# 1. Setup

This section imports the required libraries and defines the device used for training.

If a GPU is available, the notebook uses CUDA. Otherwise, it uses CPU.

In [ ]:

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 2. Data preparation

The MNIST images are originally 28 by 28 pixels.

In this notebook, the images are resized to 8 by 8 pixels and then flattened into vectors of length 64.

This means each image becomes a one-dimensional tensor with 64 values.

In [ ]:
transform_8x8_flatten = transforms.Compose([
    transforms.Resize((8, 8)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

batch_size = 64

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform_8x8_flatten
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform_8x8_flatten
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Number of training examples:", len(train_dataset))
print("Number of test examples:", len(test_dataset))


# Question 1 — DataLoaders, object types, shapes and shuffling

**Question:**  
Dataloaders `train_loader` and `test_loader` are iterators which allow access to examples and labels.

1. What is the type of objects yielded by the train and test dataloaders?  
2. What is the shape of images returned by `for images, labels in train_loader`? How do you interpret that shape?  
3. Why do dataloaders for train and test differ with respect to the option `shuffle`?  
4. Visualize some examples and labels.

In [ ]:

images, labels = next(iter(train_loader))

print("Type of images object:", type(images))
print("Type of labels object:", type(labels))
print("Shape of images tensor:", images.shape)
print("Shape of labels tensor:", labels.shape)
print("Data type of images:", images.dtype)
print("Data type of labels:", labels.dtype)


## Answer to Question 1

The `train_loader` and `test_loader` yield batches. Each batch is a pair:

`images, labels`

Both objects are PyTorch tensors.

The shape of `images` is usually:

`[64, 64]`

The first dimension is the batch size, meaning there are 64 images in the batch.  
The second dimension is the number of input features for each image. Since each MNIST image was resized to 8 by 8 pixels and then flattened, each image has:

`8 × 8 = 64`

features.

The labels tensor usually has shape:

`[64]`

This means there is one class label for each image in the batch.

The training dataloader uses `shuffle=True` because randomizing the order of training examples helps the model learn more robustly and prevents it from seeing examples in the same order at every epoch.

The test dataloader uses `shuffle=False` because we do not train on test data. For evaluation, the order does not need to be randomized, and keeping it fixed makes results easier to reproduce and inspect.

In [ ]:
examples, example_labels = zip(*[train_dataset[i] for i in range(12)])

fig, axes = plt.subplots(3, 4, figsize=(8, 7))

for i, ax in enumerate(axes.flat):
    image_2d = examples[i].reshape(8, 8)
    ax.imshow(image_2d, cmap="gray")
    ax.set_title(f"Label: {example_labels[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


# 3. Model development

The original model is a simple multilayer perceptron with one hidden layer.

Because the images are resized to 8 by 8 and flattened, the input size is 64.

The model returns 10 output scores, one for each digit class from 0 to 9.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


input_size = 8 * 8
hidden_size = 128
num_classes = 10

model = SimpleNN(input_size, hidden_size, num_classes).to(device)

print(model)


# Question 2 — Architecture and activation functions

**Question:**

1. The model architecture depends on the input data shape. The input images of MNIST are originally 28 by 28 pixels, but they have been resized to 8 by 8 pixels. Which changes are needed to use the original 28 by 28 size? Indicate two changes needed in the notebook.
2. Which change is needed in the `SimpleNN` class if the model should have two hidden layers, both with size 128?
3. What is a ReLU activation function called by `nn.ReLU`?
4. Why are nonlinear activation functions like ReLU necessary for deep learning?

## Answer to Question 2

To use the original 28 by 28 MNIST images, two changes are needed.

First, the transformation should not resize the images to 8 by 8. Instead, the image should be converted to a tensor and flattened directly.

Second, the input size of the model should be changed from:

`8 × 8 = 64`

to:

`28 × 28 = 784`

So, the model input size would become `784`.

To create a neural network with two hidden layers of size 128, the model class must include an additional linear layer. The first layer maps the input to 128 neurons, the second hidden layer maps 128 neurons to another 128 neurons, and the final output layer maps 128 neurons to 10 classes.

`nn.ReLU()` is the Rectified Linear Unit activation function. It returns zero for negative input values and returns the input itself for positive values.

Nonlinear activation functions are necessary because, without them, a neural network with several linear layers would still behave like a single linear transformation. ReLU introduces nonlinearity, allowing the network to learn complex patterns in the data.

In [ ]:
# Note on AI use:
# prompt: Help me show the minimal code changes needed for 28x28 inputs and an extra hidden layer. I added this as an explanatory example without replacing the main model used later.

transform_28x28_flatten = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

input_size_28x28 = 28 * 28

class TwoHiddenLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(TwoHiddenLayerNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

example_two_hidden_model = TwoHiddenLayerNN(
    input_size=input_size_28x28,
    hidden_size=128,
    num_classes=10
)

print(example_two_hidden_model)


# 4. Loss function and optimizer

This is a classification task with 10 possible classes.

For this reason, the loss function is `CrossEntropyLoss`.

The optimizer used here is Adam.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 5. Basic training loop

This section defines helper functions for training and evaluating the model.

The helper functions are reused later for the timing comparison and for the corrected accuracy plot.

In [ ]:
# Note on AI use:
# prompt: Help me separate the PyTorch training loop into small reusable functions.

def compute_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    return correct / total


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    batch_losses = []
    batch_accuracies = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())
        batch_accuracies.append(compute_accuracy(outputs, labels))

    return np.mean(batch_losses), np.mean(batch_accuracies)


def evaluate_accuracy(model, loader, device):
    model.eval()
    batch_accuracies = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            batch_accuracies.append(compute_accuracy(outputs, labels))

    return np.mean(batch_accuracies)


# Question 3 — `num_workers` and processing time

**Question:**  
Does processing time decrease if we add the option `num_workers=2` when defining the dataloader?

To answer this, I train the same type of model twice:

1. with `num_workers=0`;
2. with `num_workers=2`.

Then I compare the training time.

In [ ]:
# Note on AI use:
# prompt: Help me compare training time with num_workers=0 and num_workers=2.

def make_train_loader(num_workers):
    return DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )


def time_training_run(num_workers, epochs=2):
    torch.manual_seed(seed)

    timing_model = SimpleNN(
        input_size=8 * 8,
        hidden_size=128,
        num_classes=10
    ).to(device)

    timing_optimizer = optim.Adam(timing_model.parameters(), lr=0.001)
    timing_loader = make_train_loader(num_workers=num_workers)

    start_time = time.perf_counter()

    for epoch in range(epochs):
        train_one_epoch(
            timing_model,
            timing_loader,
            criterion,
            timing_optimizer,
            device
        )

    end_time = time.perf_counter()

    return end_time - start_time


timing_epochs = 2

time_workers_0 = time_training_run(num_workers=0, epochs=timing_epochs)
time_workers_2 = time_training_run(num_workers=2, epochs=timing_epochs)

timing_results = pd.DataFrame({
    "DataLoader setting": ["num_workers=0", "num_workers=2"],
    "Training epochs": [timing_epochs, timing_epochs],
    "Training time seconds": [time_workers_0, time_workers_2]
})

display(timing_results)

plt.figure(figsize=(6, 4))
plt.bar(timing_results["DataLoader setting"], timing_results["Training time seconds"])
plt.ylabel("Training time (seconds)")
plt.title("Training time comparison")
plt.show()


## Answer to Question 3

The answer depends on the execution environment.

In principle, using `num_workers=2` can reduce data loading time because data batches are prepared in parallel by worker processes.

However, for this specific notebook, the images are very small because they are resized to 8 by 8 pixels. The model is also simple. Because of that, the overhead of creating extra workers may be similar to, or even larger than, the benefit of parallel loading.

Therefore, the correct conclusion should be based on the measured times shown in the table above. If the time with `num_workers=2` is lower, then it helped. If it is similar or higher, then it did not improve performance in this case.

# 6. Correct train and validation accuracy over epochs

This section trains the model for several epochs and records both training and validation accuracy.

The important correction is that training accuracy is computed after each epoch using `model.eval()` and `torch.no_grad()`, just like validation accuracy.

This makes the comparison between train and validation curves fairer.

In [ ]:
# Note on AI use:
# prompt: Help me correct the training and validation accuracy plot so both curves are evaluated consistently after each epoch.

# A non-shuffled train loader is useful for evaluation consistency.
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Reinitialize model for this experiment
torch.manual_seed(seed)

model = SimpleNN(
    input_size=8 * 8,
    hidden_size=128,
    num_classes=10
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "validation_accuracy": []
}

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    train_loss, _ = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    train_acc = evaluate_accuracy(
        model,
        train_eval_loader,
        device
    )

    val_acc = evaluate_accuracy(
        model,
        test_loader,
        device
    )

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["validation_accuracy"].append(val_acc)

    print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f} | Validation accuracy: {val_acc:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["train_accuracy"], marker="o", label="Train accuracy")
plt.plot(history["epoch"], history["validation_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Corrected train and validation accuracy over epochs")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training loss over epochs")
plt.grid(True)
plt.show()


# Question 4 — Number of epochs and validation curve

**Question:**

1. From the visualization of the plot, are 5 epochs enough, or should the model train longer?
2. Can you find a reason for the validation curve to be consistently higher than the training curve, which in principle should not happen?
3. Correct the plot construction.

## Answer to Question 4

Five epochs are enough to show that the model is learning, but they may not be enough to obtain the best possible performance.

If the validation accuracy is still increasing at epoch 5 and the training loss is still decreasing, the model would probably benefit from training for more epochs. If the validation accuracy has already stabilized, then 5 epochs may be enough for this simple demonstration.

A reason why the validation accuracy can appear consistently higher than the training accuracy in the original notebook is the way the history was computed.

In the original training loop, training accuracy was averaged from batches during the training process. This means the model was changing while those batch accuracies were being collected. Early batches in the epoch were evaluated before later weight updates.

By contrast, validation accuracy was computed after the epoch, using the updated model in evaluation mode. Therefore, validation accuracy was being measured under more favorable and more consistent conditions.

The correction is to compute both training and validation accuracy after each epoch, using:

- `model.eval()`;
- `torch.no_grad()`;
- fixed evaluation loaders.

That is what the corrected plot above does.

# Final summary

The notebook changes made for Assignment 3 were:

1. I inspected the objects yielded by the PyTorch dataloaders and interpreted their tensor shapes.
2. I explained how to change the notebook for original 28 by 28 MNIST images.
3. I showed how to modify the neural network to include two hidden layers.
4. I compared training time with `num_workers=0` and `num_workers=2`.
5. I corrected the construction of the train and validation accuracy plot.
6. I added explanations for why ReLU and nonlinear activation functions are needed in neural networks.